In [1]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine
import dotenv
import os
import sqlite3

In [2]:
nba = pd.read_csv('nba2026.csv', low_memory=False)

In [3]:
pd.set_option('display.max_rows', 81)
nba.head(3).T

,0,1,2
game_id,401809234.0,401809234.0,401809234.0
game_date,2025-10-22,2025-10-22,2025-10-22
season_type,Regular Season,Regular Season,Regular Season
home_team_id,18.0,18.0,18.0
away_team_id,5.0,5.0,5.0
player_id,2528426.0,2580365.0,3136195.0
player_name,Jordan Clarkson,Larry Nance Jr.,Karl-Anthony Towns
position,G,F,C
team_id,18.0,5.0,18.0
min,13.0,17.0,31.0


## Database Normalization
### First normal form:

1. **All tables must have a primary key**: In this table, `game_id` and `player_id` together are unique on every row, and so they form primary key.

2. **All the data must be atomic**: We don't have any non-atomic data in this example. But suppose we have a column called `Inactives` equal to a list such as `Jalen Brunson, OG Anunoby, Mikal Bridges`. That would be an example of non-atomic data.

3. **No repeating groups problem**: We can't solve the non-atomicity problem by creating separate columns if this leads to arbitrary ordering language in the column names (for example, `Inactive1`, `Inactive2`, etc.) and if it leads to a lot of missing data (there would be an `Inactive7` which would be missing any time a team has less than 7 inactive players). Instead, we would create a new table just for the values of `Inactives` with each inactive player on a different row.

In [4]:
nba[['game_id', 'player_id']].duplicated().value_counts()

False    28716
Name: count, dtype: int64

### Functional Dependence
Let X and Y be columns in a data table. Y is functionally dependent on X if each value of X has exactly one value of Y.

That's pretty abstract. So here are some guidelines that help me:

1. This use of "function" is the exact same as the concept of a function from algebra and pre-calculus. A correspondence f(x)=y is a function if each value of x has only one associated value of y.

2. X is either a primary key, or something that should be a primary key in another table.

For example, `game_date` (Y) is functionally dependent on `game_id` (X) because one `game_id` takes place on exactly one date.

### Second normal form:
In this table the primary key is a superkey consisting of two columns: `game_id` and `player_id`. 

2NF is violated if any columns are functionally dependent on part of the primary key but not the entire primary key. This can only happen if the primary key is a superkey.

In [5]:
# First create a dataframe with a primary key that has dependecies and the columns that depends on it. 
game = nba[["game_id", "game_date", "home_team_id", "away_team_id", "season_type"]]

# Drop duplicates
game = game.drop_duplicates()
game

# Remove the dependencies (but not the primary key) from the original table
nba = nba.drop(['game_date', "game_date", "home_team_id", "away_team_id", "season_type"], axis=1)

In [6]:
# First create a dataframe with a primary key that has dependecies and the columns that depends on it. 
players = nba[["player_id", "player_name"]]

# Drop duplicates
players = players.drop_duplicates()
players

# Remove the dependencies (but not the primary key) from the original table
nba = nba.drop(['player_name'], axis=1)

nba.head(3).T

,0,1,2
game_id,401809234.0,401809234.0,401809234.0
player_id,2528426.0,2580365.0,3136195.0
position,G,F,C
team_id,18.0,5.0,18.0
min,13.0,17.0,31.0
fgm,2.0,4.0,5.0
fga,7.0,7.0,12.0
fg3m,0.0,2.0,1.0
fg3a,2.0,2.0,4.0
ftm,0.0,0.0,8.0


### Third normal form:
3NF is violated if there are "transitive dependencies", that is, functional dependence between columns when neither column is part of the primary key.

In [7]:
team = nba[['team_id', 'team_full_name', 'team_color', 'team_abbreviation']].drop_duplicates()
team

,team_id,team_full_name,team_color,team_abbreviation
0,18.0,New York Knicks,1d428a,NY
1,5.0,Cleveland Cavaliers,860038,CLE
21,24.0,San Antonio Spurs,000000,SA
22,6.0,Dallas Mavericks,0064b1,DAL
49,11.0,Indiana Pacers,0c2340,IND
52,25.0,Oklahoma City Thunder,007ac1,OKC
72,9.0,Golden State Warriors,fdb927,GS
75,7.0,Denver Nuggets,0e2240,DEN
160,13.0,Los Angeles Lakers,552583,LAL
161,10.0,Houston Rockets,ce1141,HOU


In [8]:
nba = nba.drop(['team_full_name', 'team_color', 'team_abbreviation'], axis=1)

In [ ]:
team_game = nba[['game_id', 'team_id', 'team_pts', 'team_fgm',
       'team_fga', 'team_fg3m', 'team_fg3a', 'team_ftm', 'team_fta',
       'team_oreb', 'team_dreb', 'team_ast', 'team_stl', 'team_blk',
       'team_tov', 'team_pf']].drop_duplicates()
team_game

,game_id,team_id,team_pts,team_fgm,team_fga,team_fg3m,team_fg3a,team_ftm,team_fta,team_oreb,team_dreb,team_ast,team_stl,team_blk,team_tov,team_pf
0,401809234.0,18.0,119,37,86,14,40,31,36,9,39,22,8,4,14.0,23
1,401809234.0,5.0,111,40,86,19,46,12,18,4,28,21,8,4,15.0,25
21,401809235.0,24.0,125,50,87,8,20,17,28,11,39,24,10,5,15.0,24
22,401809235.0,6.0,92,31,83,9,21,21,26,9,28,20,5,3,16.0,24
49,401809236.0,11.0,135,46,105,13,43,30,40,11,44,21,6,4,14.0,37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28645,401873342.0,18.0,109,44,85,13,36,8,14,5,35,32,5,5,8.0,22
28670,401873343.0,5.0,108,42,84,12,41,12,19,9,25,22,7,1,18.0,22
28671,401873343.0,18.0,121,43,77,11,28,24,27,8,29,27,11,4,15.0,19
28689,401873344.0,5.0,93,32,77,11,40,18,22,6,27,21,9,4,22.0,18


In [10]:
team_game = nba.drop(['team_pts', 'team_fgm',
       'team_fga', 'team_fg3m', 'team_fg3a', 'team_ftm', 'team_fta',
       'team_oreb', 'team_dreb', 'team_ast', 'team_stl', 'team_blk',
       'team_tov', 'team_pf'], axis=1)

#### Third normal form:

- games
- teams
- players
- player_game
- team_game

# dbdiagram.io

In [11]:
import pandas as pd

def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.

    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.

    Returns:
        A DBML string representing the DataFrame schema.
    """

    dbml_string = f"Table {table_name} {{\n"

    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"

    dbml_string += "}\n"
    return dbml_string

def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [13]:
print(pandas_df_to_dbml(game, "game"))

Table game {
  game_id float
  game_date varchar
  home_team_id float
  away_team_id float
  season_type varchar
}



In [14]:
print(pandas_df_to_dbml(players, "players"))

Table players {
  player_id float
  player_name varchar
}



In [17]:
print(pandas_df_to_dbml(team, "team"))

Table team {
  team_id float
  team_full_name varchar
  team_color varchar
  team_abbreviation varchar
}



In [16]:
print(pandas_df_to_dbml(team_game, "team_game"))

Table team_game {
  game_id float
  player_id float
  position varchar
  team_id float
  min float
  fgm float
  fga float
  fg3m float
  fg3a float
  ftm float
  fta float
  oreb float
  dreb float
  ast float
  stl float
  blk float
  tov float
  pf float
  pts float
  plus_minus float
  is_starter float
}



In [ ]:
""""
Ref: games. game_id < player_game.game_id
Ref: players-player_id < player_game-player_id
Ref: games. game_id < team_game.game id
Ref: teams. team_id < team_game. team id
Ref: teams. team_id < player_game.team_id
Ref: teams. team_id < games.home_team_id
Ref: teams. team_id < games.away team_id
Ref: team_game, game_id < player-gane, game_id
Ref: team_game. team_id < player_game. team_id
"""